# 3D DC Inversion — speed (GPU vs CPU) and prior (TV vs L2)

Two things in one notebook, both on the **same** synthetic 3D problem:

**A. Speed** — first-order **L-BFGS + cuDSS GPU** vs SimPEG **Gauss-Newton + Pardiso MKL** (parallel CPU).
L-BFGS fits the GPU because it uses only the gradient (exact autograd Jᵀ, cuDSS's strength) and
never the Hessian/Jvec (where a direct CPU solver reusing its factorization is unbeatable).

**B. Prior / model quality** — same L-BFGS+cuDSS inversion with **L2** (smooth) vs **TV** (sharp-edges)
regularization, both cooled to the same target misfit. Only the prior changes. TV is only possible
because the forward is differentiable (any derivable penalty works) — SimPEG's numpy GN is tied to L2/IRLS.

**Fair CPU baseline = Pardiso MKL (parallel)**, NOT SimPEG's default `SolverLU` (scipy SuperLU,
single-threaded, ~10x slower in 3D).

In [ ]:
import time
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import torch

from pydiso.mkl_solver import MKLPardisoSolver as _MKL
from SimPEG.config import SimpegConfig
from SimPEG import (maps, data, data_misfit, regularization, optimization,
                    inverse_problem, inversion, directives)
from SimPEG.electromagnetics.static import resistivity as dc
from SimPEG.electromagnetics.static.utils.static_utils import generate_dcip_sources_line
from discretize import TreeMesh
from discretize.utils import active_from_xyz

cfg = SimpegConfig(); cfg.dtype = torch.float64
HAS_CUDA = torch.cuda.is_available()
GPU_SOLVER = "cudss" if HAS_CUDA else "pcg_gpu"
print("CUDA:", HAS_CUDA, "| GPU solver:", GPU_SOLVER)

class PardisoSolver:                       # fair parallel CPU baseline
    def __init__(s, A, **k): s._s = _MKL(sp.csr_matrix(A))
    def __mul__(s, b): return s._s.solve(np.asarray(b))
    def solve(s, b, **k): return s._s.solve(np.asarray(b))
    def clean(s): pass

## 1. Parameters (move to change size / cost)

In [ ]:
DH = 25.0            # base cell (m): 40->~10k cells, 25->~27k (default), 20->bigger/slower
NOISE = 0.03
GN_MAXITER, GN_MAXITERCG = 15, 15
LBFGS_OUTER, LBFGS_MAXITER, COOL = 14, 15, 2.0
TV_EPS, ALPHA_S = 1e-3, 1e-4

## 2. Survey, mesh, true model

In [ ]:
gx = np.linspace(-600,600,1201); XX,YY = np.meshgrid(gx,gx)
topo_xyz = np.c_[XX.ravel(),YY.ravel(),np.zeros(XX.size)]
ends = [np.r_[-400.,400.,0.,0.], np.r_[-400.,400.,-200.,-200.], np.r_[-400.,400.,200.,200.]]
srcs = []
for e in ends: srcs += generate_dcip_sources_line("dipole-dipole","volt","3D",e,topo_xyz,6,80.0)
survey = dc.survey.Survey(srcs)
nb_ = lambda w: 2**int(np.round(np.log(w/DH)/np.log(2.0)))
mesh = TreeMesh([[(DH,nb_(3200.))],[(DH,nb_(3200.))],[(DH,nb_(1600.))]], x0="CCN", diagonal_balance=True)
mesh.refine_points(survey.unique_electrode_locations, padding_cells_by_level=[8,8,4], finalize=False); mesh.finalize()
active = active_from_xyz(mesh, topo_xyz); nC = int(active.sum())
cc = mesh.cell_centers[active,:]
sig = 1e-2*np.ones(nC)
sig[((cc[:,0]+200)**2+cc[:,1]**2+(cc[:,2]+150)**2)<150**2] = 1e-1   # conductor
sig[((cc[:,0]-200)**2+cc[:,1]**2+(cc[:,2]+150)**2)<150**2] = 1e-3   # resistor
m_true = np.log(1/sig)
rmap = maps.InjectActiveCells(mesh, active, 1/1e-8)*maps.ExpMap(nP=nC)
survey.drape_electrodes_on_topography(mesh, active, option="top")
print(f"mesh {mesh.n_cells} | active {nC} | nD {survey.nD}")
VMIN, VMAX = 5.0, 2000.0

## 3. Data + shared regularization operators

In [ ]:
cfg.torch_is_active = False
simd = dc.simulation.Simulation3DNodal(mesh, survey=survey, rhoMap=rmap)
dclean = simd.dpred(m_true)
np.random.seed(1)
dobs = dclean + NOISE*np.abs(dclean)*np.random.randn(len(dclean))
std = 1e-4 + NOISE*np.abs(dobs)
dcd = data.Data(survey, dobs=dobs); dcd.standard_deviation = std
mref = np.log(100.0)*np.ones(nC); nD = survey.nD; target = nD/2.0

reg = regularization.WeightedLeastSquares(mesh, active_cells=active, alpha_s=ALPHA_S,
        alpha_x=1.0, alpha_y=1.0, alpha_z=1.0, reference_model=mref)
R = sp.csr_matrix(reg.deriv2(mref))                 # L2 Hessian (constant)
rm = reg.regularization_mesh
Gx = sp.csr_matrix(rm.cell_gradient_x); Gy = sp.csr_matrix(rm.cell_gradient_y); Gz = sp.csr_matrix(rm.cell_gradient_z)
print(f"nD {nD} | target phi_d {target:.0f}")

## 4. Gauss-Newton + Pardiso MKL (CPU speed reference)

In [ ]:
def run_gn():
    cfg.torch_is_active = False
    sim = dc.simulation.Simulation3DNodal(mesh, survey=survey, rhoMap=rmap, storeJ=False)
    sim.solver = PardisoSolver
    dmis = data_misfit.L2DataMisfit(data=dcd, simulation=sim)
    opt = optimization.InexactGaussNewton(maxIter=GN_MAXITER, maxIterCG=GN_MAXITERCG)
    invProb = inverse_problem.BaseInvProblem(dmis, reg, opt)
    save = directives.SaveOutputEveryIteration(save_txt=False)
    dlist = [directives.BetaEstimate_ByEig(beta0_ratio=20.0), save,
             directives.BetaSchedule(coolingFactor=COOL, coolingRate=1),
             directives.TargetMisfit(chifact=1.0)]
    inv = inversion.BaseInversion(invProb, directiveList=dlist)
    t0 = time.time(); mrec = inv.run(mref); dt = time.time()-t0
    return dict(model=mrec, time=dt, phi_d=np.array(save.phi_d))

res_gn = run_gn()
print(f"GN-Pardiso: {res_gn['time']:.1f}s | {len(res_gn['phi_d'])} iters | phi_d {res_gn['phi_d'][-1]:.1f}")

## 5. L-BFGS + cuDSS (GPU) — L2 and TV priors

In [ ]:
def s2t(M):
    M = M.tocoo()
    return torch.sparse_coo_tensor(torch.tensor(np.vstack((M.row,M.col))),
                                   torch.tensor(M.data, dtype=torch.float64), M.shape).coalesce()
R_t, Gx_t, Gy_t, Gz_t = s2t(R), s2t(Gx), s2t(Gy), s2t(Gz)
mref_t = torch.tensor(mref); W_t = torch.tensor(1.0/std); dobs_t = torch.tensor(dobs)

def phi_m(m, kind):
    dm = m - mref_t
    if kind == "L2":
        d2 = dm.unsqueeze(1)
        return 0.5*torch.sum(d2*torch.sparse.mm(R_t, d2))
    mv = lambda G,x: torch.sparse.mm(G, x.unsqueeze(1)).squeeze(1)
    tv = (torch.sqrt(mv(Gx_t,m)**2+TV_EPS**2).sum()
        + torch.sqrt(mv(Gy_t,m)**2+TV_EPS**2).sum()
        + torch.sqrt(mv(Gz_t,m)**2+TV_EPS**2).sum())
    return ALPHA_S*0.5*torch.sum(dm**2) + tv

def run_lbfgs(kind):
    cfg.torch_is_active = True; cfg.device = ("cuda" if HAS_CUDA else "cpu")
    cfg.dtype = torch.float64; cfg.solver = GPU_SOLVER
    sim = dc.simulation.Simulation3DNodal(mesh, survey=survey, rhoMap=rmap)
    m = torch.tensor(mref.copy(), requires_grad=True)             # CPU model; solve on GPU
    with torch.no_grad():
        d0 = sim.dpred(mref_t, f=sim.fields(mref_t)); d0 = d0.cpu() if d0.is_cuda else d0
        phid0 = 0.5*torch.sum((W_t*(d0-dobs_t))**2)
        pm = phi_m(mref_t + 0.1*torch.randn(nC, dtype=torch.float64), kind)
    beta = float(phid0/(pm+1e-30))
    opt = torch.optim.LBFGS([m], lr=1.0, max_iter=LBFGS_MAXITER, history_size=10, line_search_fn="strong_wolfe")
    hist = []
    if HAS_CUDA: torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    for outer in range(LBFGS_OUTER):
        pd = [None]
        def closure():
            opt.zero_grad()
            m_in = m*1.0                                          # fresh node -> invalidate MeSigma cache
            d = sim.dpred(m_in, f=sim.fields(m_in)); d = d.cpu() if d.is_cuda else d
            loss = 0.5*torch.sum((W_t*(d-dobs_t))**2) + beta*phi_m(m, kind)
            loss.backward(); pd[0] = float(0.5*torch.sum((W_t*(d-dobs_t))**2)); return loss
        opt.step(closure); hist.append(pd[0])
        if pd[0] <= target: break
        beta /= COOL
    dt = time.time()-t0
    vram = torch.cuda.max_memory_allocated()/1e9 if HAS_CUDA else 0.0
    cfg.torch_is_active = False
    return dict(model=m.detach().cpu().numpy(), time=dt, phi_d=np.array(hist), vram=vram)

print("L-BFGS L2 ..."); res_l2 = run_lbfgs("L2")
print(f"  {res_l2['time']:.1f}s | outer {len(res_l2['phi_d'])} | phi_d {res_l2['phi_d'][-1]:.1f} | VRAM {res_l2['vram']:.2f} GB")
print("L-BFGS TV ..."); res_tv = run_lbfgs("TV")
print(f"  {res_tv['time']:.1f}s | outer {len(res_tv['phi_d'])} | phi_d {res_tv['phi_d'][-1]:.1f}")

## 6. A. Speed summary (GPU vs CPU)

In [ ]:
print(f"GN-Pardiso (CPU):   {res_gn['time']:6.1f} s   phi_d {res_gn['phi_d'][-1]:.1f}")
print(f"L-BFGS-cuDSS (GPU): {res_l2['time']:6.1f} s   phi_d {res_l2['phi_d'][-1]:.1f}   VRAM {res_l2['vram']:.2f} GB")
print(f"-> speedup {res_gn['time']/res_l2['time']:.2f}x ({'GPU wins' if res_l2['time']<res_gn['time'] else 'CPU wins'})")
print(f"   (L-BFGS-L2 vs GN model rel-diff: {np.linalg.norm(res_gn['model']-res_l2['model'])/np.linalg.norm(res_gn['model']):.2e})")

## 7. B. Recovered models — True / L2 / TV (Y=0 slice)

In [ ]:
plotting_map = maps.InjectActiveCells(mesh, active, np.nan)
def errf(ml): return np.linalg.norm(ml-m_true)/np.linalg.norm(m_true)
def pslice(ax, mlog, title):
    mesh.plot_slice(plotting_map*np.exp(mlog), ax=ax, normal="Y", grid=False,
        pcolor_opts={"cmap":"RdYlBu_r","norm":LogNorm(vmin=VMIN,vmax=VMAX)})
    ax.set_title(title); ax.set_xlim([-600,600]); ax.set_ylim([-600,0]); ax.set_xlabel("x (m)"); ax.set_ylabel("z (m)")

fig, axs = plt.subplots(3,1,figsize=(11,10))
pslice(axs[0], m_true, "True")
pslice(axs[1], res_l2['model'], f"L2  (smooth) | err {errf(res_l2['model']):.3f}")
pslice(axs[2], res_tv['model'], f"TV  (sharp)  | err {errf(res_tv['model']):.3f}")
for a in axs: fig.colorbar(a.collections[0], ax=a, label=r"$\rho$")
plt.tight_layout(); plt.show()
print("NB: global L2 error favors the L2 prior by construction; judge TV by edge sharpness / contrast (visual).")

## 8. Convergence

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
ax.semilogy(np.arange(1,len(res_gn['phi_d'])+1), res_gn['phi_d'], "o-", label="GN-Pardiso CPU")
ax.semilogy(np.arange(1,len(res_l2['phi_d'])+1), res_l2['phi_d'], "s-", label="L-BFGS-cuDSS L2")
ax.semilogy(np.arange(1,len(res_tv['phi_d'])+1), res_tv['phi_d'], "^-", label="L-BFGS-cuDSS TV")
ax.axhline(target, ls="--", c="k", lw=1, label="target")
ax.set_xlabel("outer iteration"); ax.set_ylabel(r"$\phi_d$"); ax.legend(); ax.set_title("Data misfit")
plt.tight_layout(); plt.show()